
# 14 WT-plus 原生回放对账与前向观察

> **本章要解决的问题**：回测报告说赚了 1 万，原生回放只赚 4 千，差在哪？为什么“回放完成 40 天”不能拿历史数据凑？部分成交、撤单、拒单，怎样对上持仓和现金？

## 本章导读

第12章讲“因子怎样进 WT-plus”，这一章讲“进了 WT-plus 之后，账怎样对上”。原生回放不是“再跑一次回测”，而是用 WT-plus 的撮合、费用、持仓、账户语义，把策略在历史行情上重新执行一遍。


**WT-plus 是什么**：WT 是你改造后的 WonderTrader：C++ 核心 [WonderTrader-plus](https://github.com/newbigdeng/WonderTrader-plus) + Python 封装 [wtpy-plus](https://github.com/newbigdeng/wtpy-plus)。原生回放是这套平台的执行/账户语义，不是普通回测。

本章分六节：

1. 回放的两个目标：复现执行、前向观察；
2. 成交瀑布：中间价 → 盘口 → 滑点 → 费用；
3. 账户路径：本金、保证金、回撤、拒单；
4. 部分成交、撤单、持仓对账；
5. 前向观察：冻结、完整交易日、N/40；
6. 对账表与差异归因。

### 学习目标

- 能把中间价毛利拆成可执行、滑点、费用、净利。
- 能对账户路径、部分成交、持仓做对账。
- 能数完整交易日和前向进度。
- 能写一张对账表，并按价格/数量/费用归因差异。

### 先修

第01章的成本/乘数；第12章的因子流水线；会 pandas 的 merge/groupby。

---

### 环境准备



## 本章知识结构图

<svg style="max-width:100%;height:auto" xmlns="http://www.w3.org/2000/svg" width="800" height="450" viewBox="0 0 800 450" role="img" aria-label="14 WT-plus 回放对账与前向观察" font-family="Microsoft YaHei, SimHei, sans-serif">
<title>14 WT-plus 回放对账与前向观察｜知识图</title>
<desc>本章主线：中间价毛利 ≠ 到手净利；滑点、手续费、部分成交逐笔扣掉，才谈得上前向观察。</desc>
<rect x="0" y="0" width="800" height="450" fill="#FFFFFF"/>
<defs><marker id="a14" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="7" markerHeight="7" orient="auto-start-reverse"><path d="M0,0 L10,5 L0,10 Z" fill="#1B3A5C"/></marker><marker id="ao14" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="7" markerHeight="7" orient="auto-start-reverse"><path d="M0,0 L10,5 L0,10 Z" fill="#E07B39"/></marker><marker id="ag14" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="7" markerHeight="7" orient="auto-start-reverse"><path d="M0,0 L10,5 L0,10 Z" fill="#6E7B8B"/></marker></defs>
<rect x="40" y="19" width="46" height="29" fill="#E07B39" rx="8"/>
<text x="63" y="39" font-size="15" fill="#FFFFFF" text-anchor="middle" font-weight="bold">14</text>
<text x="98" y="41" font-size="22" fill="#1B3A5C" font-weight="bold">WT-plus 回放对账与前向观察</text>
<line x1="40" y1="57" x2="760" y2="57" stroke="#E6EFF7" stroke-width="2" stroke-linecap="round"/>
<line x1="40" y1="407" x2="760" y2="407" stroke="#E6EFF7" stroke-width="1.6" stroke-linecap="round"/>
<rect x="40" y="418" width="4" height="10" fill="#E07B39" rx="1"/>
<text x="52" y="428" font-size="13" fill="#6E7B8B">本章主线：中间价毛利 ≠ 到手净利；滑点、手续费、部分成交逐笔扣掉，才谈得上前向观察。</text>
<rect x="41" y="84" width="165" height="42" fill="#E6EFF7" stroke="#1B3A5C" stroke-width="1.3" rx="14"/>
<text x="123.5" y="94.3" font-size="9.2" fill="#1B3A5C" text-anchor="middle">中间价毛利</text>
<text x="123.5" y="107.5" font-size="10.5" fill="#6E7B8B" text-anchor="middle">10,000</text>
<rect x="224" y="84" width="165" height="42" fill="#FFFFFF" stroke="#E07B39" stroke-width="1.3" rx="14"/>
<text x="306.5" y="94.3" font-size="9.2" fill="#1B3A5C" text-anchor="middle">− 滑点</text>
<text x="306.5" y="107.5" font-size="10.5" fill="#6E7B8B" text-anchor="middle">−2,400</text>
<rect x="407" y="84" width="165" height="42" fill="#FFFFFF" stroke="#E07B39" stroke-width="1.3" rx="14"/>
<text x="489.5" y="94.3" font-size="9.2" fill="#1B3A5C" text-anchor="middle">− 手续费 / 冲击</text>
<text x="489.5" y="107.5" font-size="10.5" fill="#6E7B8B" text-anchor="middle">−2,000</text>
<rect x="590" y="84" width="165" height="42" fill="#E6EFF7" stroke="#1B3A5C" stroke-width="1.3" rx="14"/>
<text x="672.5" y="94.3" font-size="9.2" fill="#1B3A5C" text-anchor="middle">= 净利与对账</text>
<text x="672.5" y="107.5" font-size="10.5" fill="#6E7B8B" text-anchor="middle">5,000</text>
<path d="M 209 105 L 221 105" fill="none" stroke="#6E7B8B" stroke-width="1.2" stroke-linecap="round" stroke-linejoin="round" marker-end="url(#ag14)"/>
<path d="M 392 105 L 404 105" fill="none" stroke="#6E7B8B" stroke-width="1.2" stroke-linecap="round" stroke-linejoin="round" marker-end="url(#ag14)"/>
<path d="M 575 105 L 587 105" fill="none" stroke="#6E7B8B" stroke-width="1.2" stroke-linecap="round" stroke-linejoin="round" marker-end="url(#ag14)"/>
<rect x="79" y="160" width="92" height="160" fill="#E6EFF7" stroke="#1B3A5C" stroke-width="1.3" rx="4"/>
<text x="125" y="153" font-size="9.5" fill="#1B3A5C" text-anchor="middle" font-weight="bold">10,000</text>
<text x="125" y="340" font-size="8.8" fill="#6E7B8B" text-anchor="middle">中间价毛利</text>
<rect x="195" y="160" width="92" height="38.4" fill="#E07B39" rx="4"/>
<text x="241" y="153" font-size="9.5" fill="#E07B39" text-anchor="middle" font-weight="bold">−2,400</text>
<text x="241" y="340" font-size="8.8" fill="#6E7B8B" text-anchor="middle">滑点</text>
<line x1="171" y1="160" x2="195" y2="160" stroke="#6E7B8B" stroke-width="1" stroke-linecap="round" stroke-dasharray="3 2"/>
<rect x="311" y="198.4" width="92" height="12.8" fill="#E07B39" rx="4"/>
<text x="357" y="191.4" font-size="9.5" fill="#E07B39" text-anchor="middle" font-weight="bold">−800</text>
<text x="357" y="340" font-size="8.8" fill="#6E7B8B" text-anchor="middle">手续费</text>
<line x1="287" y1="198.4" x2="311" y2="198.4" stroke="#6E7B8B" stroke-width="1" stroke-linecap="round" stroke-dasharray="3 2"/>
<rect x="427" y="211.2" width="92" height="19.2" fill="#E07B39" rx="4"/>
<text x="473" y="204.2" font-size="9.5" fill="#E07B39" text-anchor="middle" font-weight="bold">−1,200</text>
<text x="473" y="340" font-size="8.8" fill="#6E7B8B" text-anchor="middle">部分成交</text>
<line x1="403" y1="211.2" x2="427" y2="211.2" stroke="#6E7B8B" stroke-width="1" stroke-linecap="round" stroke-dasharray="3 2"/>
<rect x="543" y="230.4" width="92" height="9.6" fill="#E07B39" rx="4"/>
<text x="589" y="223.4" font-size="9.5" fill="#E07B39" text-anchor="middle" font-weight="bold">−600</text>
<text x="589" y="340" font-size="8.8" fill="#6E7B8B" text-anchor="middle">冲击</text>
<line x1="519" y1="230.4" x2="543" y2="230.4" stroke="#6E7B8B" stroke-width="1" stroke-linecap="round" stroke-dasharray="3 2"/>
<rect x="659" y="240" width="92" height="80" fill="#1B3A5C" stroke="#1B3A5C" stroke-width="1.3" rx="4"/>
<text x="705" y="233" font-size="9.5" fill="#1B3A5C" text-anchor="middle" font-weight="bold">5,000</text>
<text x="705" y="340" font-size="8.8" fill="#6E7B8B" text-anchor="middle">净利</text>
<line x1="635" y1="240" x2="659" y2="240" stroke="#6E7B8B" stroke-width="1" stroke-linecap="round" stroke-dasharray="3 2"/>
<line x1="79" y1="320" x2="751" y2="320" stroke="#6E7B8B" stroke-width="1.2" stroke-linecap="round"/>
<text x="402" y="372" font-size="9.2" fill="#6E7B8B" text-anchor="middle">毛利是研究口径，净利才是账户口径</text>
<text x="705" y="386" font-size="8.8" fill="#1B3A5C" text-anchor="middle" font-weight="bold">对账：账户口径</text>
</svg>

> **图**：本章主线：中间价毛利 ≠ 到手净利；滑点、手续费、部分成交逐笔扣掉，才谈得上前向观察。


In [1]:

import matplotlib.pyplot as plt

plt.rcParams["font.sans-serif"] = ["Microsoft YaHei", "SimHei", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False
print("matplotlib 中文字体已设置")


matplotlib 中文字体已设置



## 在量化研究中的位置

- **谁用**：交易员、执行、中台、策略开发。
- **输入**：订单/成交/账户/日志、行情。
- **输出**：成交瀑布、账户路径、对账表、前向进度。
- **在流程的哪一步**：回放与上线前向观察。
- **失败模式**：中间价当成交、持仓不守恒、用回测凑前向天数。
- **最小工作流**：回放 → 成交瀑布 → 账户/部分成交 → 对账 → 完整交易日计数。

---
## 1. 回放的两个目标

### 1.1 复现执行

给定历史行情、策略参数、费用/账户语义，WT-plus 原生回放逐笔撮合，输出：

- 每笔委托/成交；
- 手续费、滑点；
- 持仓、现金、权益；
- 拒单/撤单原因。

它回答“如果当时真的这么跑，账会是什么样”。

### 1.2 前向观察

候选冻结后，从**下一个完整交易日**开始，逐日积累“未参与研究”的样本。它不是回测，不能用历史数据补；它是“冻结之后真实发生”的记录。

### 1.3 两个常见混淆

- 把回测收益当“前向观察”；
- 把“回放跑完”当“策略有效”。

**回放是复现，前向是验证；两者不能互相替代。**

**常见错误**：

1. 用回测数据补前向观察天数。
2. 只报权益曲线，不报逐笔成交/费用。
3. 冻结日算错，把冻结当天算进完整交易日。

**停下来想一想**：为什么前向观察不能用历史回放凑？

<details><summary>对照你的思路</summary>

历史回放的数据已经参与过研究/调参，不是“未见过”的样本；用它凑前向天数，等于把样本内当样本外。前向观察要求“冻结之后、真实发生、未参与选择”的交易日。

</details>

---



> **真实回放入口**：`factor_native_audit.audit_version` 读 `native-trials.json`，要求完整试验族、同一版本/数据快照、40 个共同完整交易日；细节见第37章 §7。

## 2. 成交瀑布：中间价 → 盘口 → 滑点 → 费用

### 2.1 四层

| 层 | 计算 | 差额 |
|---|---|---|
| 中间价毛利 | (mid_out − mid_in) × 乘数 | 基准 |
| 盘口可执行 | (bid/ask) × 乘数 | 价差损耗 |
| 加滑点 | 实际成交价 | 额外滑点 |
| 扣手续费 | 成交额/手数 | 费用 |
| 净利 | 最后一行 | — |

### 2.2 手算

做多，开盘盘口 99/101，平仓盘口 103/105，乘数 10。

- 中间价：100 → 104，毛利 40；
- 可执行：买 101、卖 103，毛利 20；
- 价差损耗 20；
- 额外滑点 1 点 = 10；
- 手续费 6；
- 净利 4。

**“中间价赚 40”到“账户赚 4”，中间有 36 元的摩擦。**

### 2.3 代码：成交瀑布



In [2]:

mid_gain = (104 - 100) * 10
executable = (103 - 101) * 10
slippage = 10
fee = 6
net = executable - slippage - fee

print("中间价毛利", mid_gain)
print("价差损耗", mid_gain - executable)
print("可执行毛利", executable)
print("净利", net)


中间价毛利 40
价差损耗 20
可执行毛利 20
净利 4



**读数与应用**：中间价 40、价差损耗 20、可执行 20、滑点 10、费用 6、净利 4。报告要把这四层分开，不能只报净利。

**常见错误**：

1. 用中间价算收益，用市价单成交。
2. 价差和滑点重复扣。
3. 只算单边费用。

**停下来想一想**：为什么“中间价策略”在实盘常常失效？

<details><summary>对照你的思路</summary>

中间价不是可成交价；买入要付卖价，卖出只能拿买价。价差越宽、流动性越差，中间价与可成交价的差距越大。很多“中间价有效”的因子，扣掉价差后就没有了。

</details>

---



## 3. 账户路径：本金、保证金、回撤、拒单

### 3.1 本金会改变成交集合

同样的信号，本金不同：

- 保证金不够 → 拒单；
- 回撤超限 → 停止开仓；
- 手数取整 → 成交数量不同。

所以“固定所有原始入场、只改价格”和“按账户状态重新决策”是两种实验，不能混。

### 3.2 手算

教学门槛：权益至少 100 元才开仓，每次占用 15 元。

- 从 110 开始：开 A 后 95，后面 B/C 被拒；
- 从 130 开始：A→115、B→100、C→85，三笔都能开。

### 3.3 代码：两条账户路径



In [3]:

for start in [110, 130]:
    equity = start
    filled = []
    for sig in ["A", "B", "C"]:
        if equity >= 100:
            filled.append(sig)
            equity -= 15
    print("起始", start, "成交", filled, "剩余", equity)


起始 110 成交 ['A'] 剩余 95
起始 130 成交 ['A', 'B', 'C'] 剩余 85



**读数与应用**：起始 110 只成交 A；起始 130 成交 A/B/C。同样的信号，本金不同，成交集合不同。

**常见错误**：

1. 用固定手数/固定资金回测，再假设“资金无限”。
2. 忽略保证金/回撤拒单。
3. 把“拒单”当“没信号”。

**停下来想一想**：为什么“本金越大，亏损越多”不一定是策略变差？

<details><summary>对照你的思路</summary>

本金大，能开更多手/更多笔，亏损的绝对金额自然更大；要看收益率/回撤比例，不是绝对金额。但本金大也可能触发不同的保证金/风控路径，成交集合变了，不能直接比较。

</details>

---



## 4. 部分成交、撤单、持仓对账

### 4.1 三个数

- **委托数量**：你请求的数量；
- **成交数量**：实际撮合的数量；
- **剩余数量**：还没成交的数量。

**持仓只由成交数量改变**，不由委托数量改变。

### 4.2 事件序列

| 事件 | 剩余 | 持仓 |
|---|---:|---:|
| 申请买 3 手 | 3 | 0 |
| 成交 1 手 | 2 | +1 |
| 成交 1 手 | 1 | +2 |
| 撤余单 1 手 | 0 | +2 |
| 平仓 2 手 | 0 | 0 |

若把“撤余单”当成“取消整张订单”，就会把已成交的 2 手也抹掉。

### 4.3 代码：事件对账



In [4]:

position = 0
remaining = 3

for event, qty in [("开仓成交", 1), ("开仓成交", 1), ("撤余单", 1), ("平仓成交", 2)]:
    if event == "开仓成交":
        remaining -= qty
        position += qty
    elif event == "撤余单":
        remaining -= qty
    else:
        position -= qty
    print(event, "剩余", remaining, "持仓", position)


开仓成交 剩余 2 持仓 1
开仓成交 剩余 1 持仓 2
撤余单 剩余 0 持仓 2
平仓成交 剩余 0 持仓 0



**读数与应用**：最终剩余 0、持仓 0。每一步都要用“成交数量”更新持仓，用“撤单数量”更新剩余。

**常见错误**：

1. 用委托数量更新持仓。
2. 撤单把已成交部分也抹掉。
3. 不记录“剩余数量”，部分成交后重复下单。

**停下来想一想**：部分成交后撤单，账户里应该剩几手？

<details><summary>对照你的思路</summary>

已成交多少，就剩多少持仓；撤单只撤“未成交的剩余”。例如申请 3 手、成交 2 手、撤 1 手，持仓是 2 手，不是 0 手。

</details>

---



## 5. 前向观察：冻结、完整交易日、N/40

### 5.1 冻结

候选冻结后，记录冻结时刻、版本、参数。从**冻结后的下一个完整交易日**开始，逐日积累“未参与研究”的记录。

### 5.2 完整交易日

一个交易日必须包含完整交易时段（含夜盘归属）。若冻结发生在夜盘中间，当天不算完整；从下一个交易日开始。

### 5.3 N/40

前向观察常用“完成 N 个完整交易日”作为门槛。N 是预先写死的，不是“看结果再决定”。

### 5.4 代码：完整交易日计数



In [5]:

from datetime import date

complete_days = [date(2026, 11, 2), date(2026, 11, 3)]
print("已完成完整交易日", len(complete_days))
print("进度", f"{len(complete_days)}/40")


已完成完整交易日 2
进度 2/40



**读数与应用**：完成 2 天，进度 2/40。真实系统要从交易日历/回放记录里数，不要手工填。

**常见错误**：

1. 把冻结当天算进完整交易日。
2. 用回测数据补天数。
3. 看到结果好就提前结束观察。

**停下来想一想**：为什么前向观察要预先写死 N？

<details><summary>对照你的思路</summary>

如果“看结果再决定观察多久”，就会在好运时停、坏运时等，产生选择偏差。预先写死 N，才能保证“冻结后的样本”不被结果影响。

</details>

---



## 6. 对账表与差异归因

### 6.1 对账表

本地账 vs 原生账，按交易 ID 对齐：

| trade | 本地净利 | 原生净利 | 差异 |
|---|---:|---:|---:|
| T1 | 10 | 10 | 0 |
| T2 | −5 | −3 | −2 |

差异要归因：价格、数量、费用、滑点、拒单、时间。

### 6.2 分组汇总

按 setup/tag 汇总，能看出“哪类交易差异最大”：

- breakout；
- extended；
- range_or_turn；
- protect。

### 6.3 代码：差异表



In [6]:

import pandas as pd

local = pd.DataFrame({"trade": ["T1", "T2"], "net": [10, -5]})
native = pd.DataFrame({"trade": ["T1", "T2"], "net": [10, -3]})

diff = local.merge(native, on="trade", suffixes=("_local", "_native"))
diff["diff"] = diff["net_local"] - diff["net_native"]
print(diff)


  trade  net_local  net_native  diff
0    T1         10          10     0
1    T2         -5          -3    -2



**读数与应用**：T1 一致，T2 差 2 元。差异可能来自成交价、手续费、滑点或数量；要逐笔查，不要只报一个总数。

### 6.4 对账检查清单

- [ ] 每笔委托/成交/撤单都有 ID；
- [ ] 持仓 = 累计成交，不是累计委托；
- [ ] 手续费/滑点按双边；
- [ ] 现金/权益逐日对；
- [ ] 差异按价格/数量/费用/拒单归因；
- [ ] 前向观察按完整交易日计数。

**常见错误**：

1. 只对总权益，不对逐笔。
2. 差异归因写“滑点”，但没算价差。
3. 把“回放完成”当“前向观察完成”。

**停下来想一想**：本地账和原生账差 2 元，怎么定位？

<details><summary>对照你的思路</summary>

先按交易 ID 对齐，比较成交价、数量、手续费、滑点；再看这笔交易是否部分成交/拒单/延迟。差异要落到“哪一腿、哪一项”，不能只写“系统差异”。

</details>

---



## 本章小结

### 一张表

| 概念 | 公式/做法 | 用途 |
|---|---|---|
| 成交瀑布 | 中间价→盘口→滑点→费用 | 解释净利 |
| 账户路径 | 本金/保证金/回撤/拒单 | 成交集合 |
| 持仓对账 | 持仓=累计成交 | 部分成交 |
| 前向观察 | 冻结后完整交易日 | N/40 |
| 差异归因 | 价格/数量/费用/拒单 | 对账 |

### 八句话

1. 回放是复现，前向是验证。
2. 中间价毛利不等于可执行净利。
3. 价差、滑点、手续费要分开记。
4. 本金/保证金会改变成交集合。
5. 持仓只由成交数量改变。
6. 撤单只撤未成交的剩余。
7. 前向观察不能用回测凑。
8. 对账要逐笔归因，不能只报总数。

---

## 本章习题

**1.** 中间价毛利 40、价差损耗 20、滑点 10、手续费 6。求净利。

**2.** 起始 110、门槛 100、每次占用 15。能开几笔？

**3.** 申请买 3 手，成交 2 手，撤 1 手。持仓是多少？

**4.** 为什么“回放完成”不等于“前向观察完成”？

**5.** 冻结当天为什么不算完整交易日？

**6.** 本地净利 10、原生 8。差异可能来自哪四项？

**7.** 前向观察为什么要预先写死 N？

**8.** 部分成交后撤单，剩余数量和持仓分别怎么变？

<details><summary>参考答案</summary>

**1.** $40-20-10-6=4$。

**2.** 起始 110：开 1 笔后 95，后面拒；只能开 1 笔。

**3.** 持仓 2 手。

**4.** 回放用历史数据复现；前向观察要求冻结后真实发生的完整交易日，未参与研究。

**5.** 冻结可能发生在交易时段中间，当天不是完整交易日；从下一个完整交易日开始。

**6.** 成交价、数量、手续费、滑点（还可能拒单/延迟）。

**7.** 防止“看结果再决定观察多久”的选择偏差。

**8.** 剩余数量减少撤单数量；持仓不变（只由成交改变）。

</details>

---

## 参考资料

- WonderTrader-plus: https://github.com/newbigdeng/WonderTrader-plus
- wtpy-plus: https://github.com/newbigdeng/wtpy-plus
- 第37章（工具链与回放的位置）。
- 本项目 `F:\0_MyLog\_教材化\进度.md`。
